# Interactive session with Spark to explore Bronze results

This notebook uses the project virtual environment and repo-local paths only.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 8g --executor-memory 8g pyspark-shell"

repo_root = Path.cwd().resolve()
if not (repo_root / "bronze").exists():
    repo_root = Path("/home/dcamacho/dev/ProjectData").resolve()

source_sample_dir = repo_root / "sample_data"
source_dir = repo_root / "data/exports/projectA"
source_dir_B = repo_root / "data/exports/projectB"
table_path = repo_root / "_tmp" / "bronze_pid_documents"
spark_warehouse = repo_root / "spark-warehouse"
metastore_db = repo_root / "metastore_db"

# Remove stale Spark/Python overrides from previous runs
for key in ["PYTHONPATH", "SPARK_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON"]:
    os.environ.pop(key, None)
    
sys.path = [p for p in sys.path if "/opt/spark" not in p]

print("repo_root =", repo_root)
print("source_sample_dir =", source_sample_dir)
print("source_dir =", source_dir)
print("source_dir_B =", source_dir_B)
print("table_path =", table_path)
print("spark_warehouse =", spark_warehouse)
print("metastore_db =", metastore_db)
print("python_executable =", sys.executable)

print("Python:", sys.executable)
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

Empecemos asegurando que la carpeta donde probaremos el almacenamiento de la tabla sin metastore

In [ ]:
import shutil

# Aseguramos que la ruta sea un objeto Path
path_a_eliminar = table_path

if path_a_eliminar.exists() and path_a_eliminar.is_dir():
    # Borra la carpeta y todo lo que tiene adentro de forma recursiva
    shutil.rmtree(path_a_eliminar)
    print(f"La carpeta {path_a_eliminar.name} fue eliminada con éxito.")
else:
    print("La carpeta no existe o ya había sido eliminada.")

Verifiquemos la version de spark que estamos correiendo (debe ser la del virtual environment)

In [ ]:
import pyspark
print(pyspark.__file__)
print(pyspark.__version__)

Usamos el comando cli para la ingesta de datos en bronze, tomando los datos de muestra (sample), y guardando los resultados en archivos que representan la tabla (sin metastore). 

In [ ]:
python_executable = globals().get("python_executable", globals().get("current_python", sys.executable))
source_sample_dir_str = str(source_sample_dir)
source_dir_str = str(source_dir)
table_path_str = str(table_path)


In [ ]:

!{python_executable} -m bronze.cli ingest \
    --source-dir "{source_sample_dir_str}" \
    --table-path "{table_path_str}" \
    --project-code Sample

In [ ]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"


Inicio de sesión en Spark en local para uso didactico, usaremos delta pipeline adicionalmente, y le asignamos 8GB de RAM al driver y executors

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Interactive_Bronze_Exploration")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark ready:", spark.version)

Verifiquemos como quedaron escritos los datos en nuestra tabla

In [ ]:
# Load the table created by the CLI ingest run
path = str(table_path)
df = spark.read.format("delta").load(path)
# print("Rows:", df.count())
df.show(50,truncate=100)

In [ ]:
# Show a compact preview
visible_columns = [
    "document_number",
    "drawing_revision",
    "project_code",
    "source_filename",
    "file_size_bytes",
    "ingested_at",
    "content_text",
]

preview = df.select(*[c for c in visible_columns if c in df.columns]).limit(10).toPandas()
preview

## Delta history

Las tablas delta guardan su propia historia, luego de cargar la primera vez vemos que solo tiene una versión.

In [ ]:
from delta.tables import DeltaTable

try:
    delta_table = DeltaTable.forPath(spark, str(table_path))
    history_df = delta_table.history()
    history_df.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)
except Exception as e:
    print(f"Unable to read Delta history: {e}")

Ahora cargamos los datos del proyecto A.

In [ ]:
python_executable = globals().get("python_executable", globals().get("current_python", sys.executable))
source_sample_dir_str = str(source_sample_dir)
source_dir_str = str(source_dir)
table_path_str = str(table_path)

!{python_executable} -m bronze.cli ingest \
    --source-dir "{source_dir_str}" \
    --table-path "{table_path_str}" \
    --project-code A

Si volvemos a verificar nuestra tabla, vemos los nuevos archivos del proyecto A, junto a los archivos de test (Sample)

In [ ]:
preview = df.select(*[c for c in visible_columns if c in df.columns]).limit(10).toPandas()
preview

In [ ]:
try:
    delta_table = DeltaTable.forPath(spark, str(table_path))
    history_df = delta_table.history()
    history_df.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)
except Exception as e:
    print(f"Unable to read Delta history: {e}")

En una tabla delta, podemos inspeccionar los datos en versiones anteriores, en este caso en la version 0 (versionAsOf) teníamos solo 6 archivos

In [ ]:
# Optional: time-travel example for version 0 if it exists
try:
    version_zero = spark.read.format("delta").option("versionAsOf", 0).load(str(table_path))
    print("Rows in version 0:", version_zero.count())
    version_zero.show(20, truncate=100)
except Exception as e:
    print(f"Version travel is unavailable yet: {e}")
finally:
    # keep the session alive until you explicitly stop it
    pass

Podemos parar la sesión de spark.

In [ ]:
spark.stop()

# Metadata Store

In [ ]:
# Aseguramos que la ruta sea un objeto Path
path_a_eliminar = metastore_db

if path_a_eliminar.exists() and path_a_eliminar.is_dir():
    # Borra la carpeta y todo lo que tiene adentro de forma recursiva
    shutil.rmtree(path_a_eliminar)
    print(f"La carpeta {path_a_eliminar.name} fue eliminada con éxito.")
else:
    print("La carpeta no existe o ya había sido eliminada.")
    
path_a_eliminar = spark_warehouse

if path_a_eliminar.exists() and path_a_eliminar.is_dir():
    # Borra la carpeta y todo lo que tiene adentro de forma recursiva
    shutil.rmtree(path_a_eliminar)
    print(f"La carpeta {path_a_eliminar.name} fue eliminada con éxito.")
else:
    print("La carpeta no existe o ya había sido eliminada.")

Continuamos ahora probando la ingesta de datos a la capa bronze usando el metastore. En este comando cli se utiliza el hive store activado, lo cual permite salvar los datos dentro del schema bronze, que alberga pid_documents (bronze.pid_documents).

In [ ]:
!{python_executable} -m bronze.cli ingest \
    --source-dir {source_dir} \
    --table bronze.pid_documents

Iniciamos una sesión para explorar el resultado. En este caso activando Hive (nuestro metadata store db)

In [ ]:
spark = (
    builder
    .appName("MetastoreApp")
    .master("local[*]")
    .enableHiveSupport()  # Required so the metastore_db is persisted
    .getOrCreate()
)

Cuando se usa el metadata store, le damos funcionalidades de una base de datos a spark. Tenemos ahora un namespace llamado bronze, una tabla pid_documents

In [ ]:
spark.sql("SHOW TABLES in bronze").show()

Tenemos en el metastore la gestion de los metadatos de la tabla bronze.pid_documents.

In [ ]:
spark.sql("DESCRIBE EXTENDED bronze.pid_documents").show(200, truncate=False)

Podemos acceder a la tabla de manera intuitiva.

In [ ]:
df = spark.table("bronze.pid_documents")
df.show(50, truncate=100)


In [ ]:
# O directamente en SQL:
spark.sql("SELECT * FROM bronze.pid_documents WHERE header_parse_ok = true")


In [ ]:
spark.stop()

# Capa Silver

1) (once) land the samples into Bronze if you haven't already

2) reconstruct Bronze -> Silver


In [ ]:
!{python_executable} -m silver.cli reconstruct --bronze-table bronze.pid_documents --silver-schema silver

Iniciemos sesión de spark para hacer algunas consultas de verificación:

In [ ]:
spark = (
    builder
    .appName("SilverApp")
    .master("local[*]")
    .enableHiveSupport()  # Required so the metastore_db is persisted
    .getOrCreate()
)

Output is JSON row counts per table. Inspect:

I built run() to accept an existing session

In [ ]:
from silver.config import SilverConfig
from silver.spark_job import run

counts = run(SilverConfig(), spark=spark)   # writes into THIS session's metastore
print(counts)

spark.table("silver.silver_segments").show(6, False)   # now visible immediately

In [ ]:
from pyspark.sql import functions as F
# tags stamped + oracle carried (but quarantined — nothing computed on it)
spark.table("silver.silver_segments").select(
    "seg_tag","fluid","piping_materials_class","src_turnover","project_code").show(6, False)
# edge provenance + direction sanity
spark.table("silver.silver_connections").groupBy("derived","flow_sense").count().show()
# component class mix (valves, fittings, …) and valve count
spark.table("silver.silver_components").groupBy("is_valve").count().show()
spark.table("silver.silver_components").groupBy("component_class").count().orderBy(F.desc("count")).show(20, False)
# lineage present on every row
spark.table("silver.silver_components").select("bronze_id","source_format","drawing_number").distinct().show(10, False)

In [ ]:
spark.table("silver.silver_segments").select("seg_tag","fluid","src_turnover").show()  # oracle carried, quarantined

In [ ]:
spark.sql("SHOW TABLES in silver").show()

# Project B Ingestion 

In [1]:
import os, sys
os.environ.pop("SPARK_HOME", None)
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if "/opt/spark" not in p)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]
assert "pyspark" not in sys.modules, "pyspark already loaded — RESTART kernel and run this FIRST"

from bronze.spark_session import get_spark          # first thing to touch pyspark
spark = get_spark()
import pyspark
print(pyspark.__file__, "| Spark", spark.version)   # want .venv + 3.5.1

your 131072x1 screen size is bogus. expect trouble
26/09/01 20:39:23 WARN Utils: Your hostname, DC01NNCOL resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/01 20:39:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c258301c-c059-4b06-bb5e-e6820696bcf7;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 170ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/__init__.py | Spark 3.5.1


In [ ]:
import os, pyspark
print("pyspark:", pyspark.__file__)      # want .../.venv/... NOT /opt/spark
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

In [ ]:
import os, sys
os.environ.pop("SPARK_HOME", None)                        # ignore system /opt/spark (Spark 4)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]

In [ ]:
import os, sys
os.environ.pop("SPARK_HOME", None)                        # ignore system /opt/spark (Spark 4)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]

from bronze.spark_session import get_spark                # extract the new tarball first
spark = get_spark()                                        # pin is built in now — no extra_conf needed

import pyspark
print("pyspark:", pyspark.__file__)                        # must be under .venv, NOT /opt/spark
print("spark version:", spark.version)                     # must be 3.5.1

# register existing Delta dirs into the pinned catalog
WH = "/home/dcamacho/dev/ProjectData/spark-warehouse"
for db, tbls in [("bronze", ["pid_documents"]),
                 ("silver", ["silver_components","silver_segments","silver_connections","silver_equipment"])]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db}")
    for t in tbls:
        loc = f"{WH}/{db}.db/{t}"
        if os.path.isdir(f"{loc}/_delta_log"):
            spark.sql(f"CREATE TABLE IF NOT EXISTS {db}.{t} USING DELTA LOCATION 'file:{loc}'")

spark.table("bronze.pid_documents").groupBy("source_format").count().show()

In [2]:
import os
REPO = "/home/dcamacho/dev/ProjectData"
WAREHOUSE_DIR = f"{REPO}/spark-warehouse"
METASTORE_DB  = f"{REPO}/metastore_db"

from bronze.spark_session import get_spark
spark = get_spark(extra_conf={
    "spark.sql.warehouse.dir": f"file:{WAREHOUSE_DIR}",
    "spark.hadoop.javax.jdo.option.ConnectionURL":
        f"jdbc:derby:;databaseName={METASTORE_DB};create=true",
})

# register the existing Delta dirs into this pinned catalog (idempotent, no data moved)
for db, tbls in [("bronze", ["pid_documents"]),
                 ("silver", ["silver_components","silver_segments","silver_connections","silver_equipment"])]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db}")
    for t in tbls:
        loc = f"{WAREHOUSE_DIR}/{db}.db/{t}"
        if os.path.isdir(f"{loc}/_delta_log"):
            spark.sql(f"CREATE TABLE IF NOT EXISTS {db}.{t} USING DELTA LOCATION 'file:{loc}'")

spark.sql("SHOW TABLES IN bronze").show()
spark.sql("SHOW TABLES IN silver").show()
spark.table("bronze.pid_documents").groupBy("source_format").count().show()

26/09/01 20:40:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/09/01 20:40:26 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/01 20:40:26 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/01 20:40:28 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/09/01 20:40:28 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dcamacho@127.0.1.1
26/09/01 20:40:29 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|   bronze|pid_documents|      false|
+---------+-------------+-----------+

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|   silver| silver_components|      false|
|   silver|silver_connections|      false|
|   silver|  silver_equipment|      false|
|   silver|   silver_segments|      false|
+---------+------------------+-----------+



26/09/01 20:40:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+-----+
|source_format|count|
+-------------+-----+
|     POSTPROC|    5|
|        DEXPI|    4|
+-------------+-----+



In [ ]:
!{python_executable} -m bronze.cli ingest \
    --source-dir {source_dir_B} \
    --table bronze.pid_documents

In [ ]:
from bronze.spark_session import get_spark
spark = get_spark()

In [ ]:
import os, glob

# 1) find the warehouse the CLI wrote into, and Bronze's path
wh = sorted(set(glob.glob(os.path.expanduser("~/dev/ProjectData/**/spark-warehouse"), recursive=True)))
print("warehouses found:", wh)
WH = next(w for w in wh if os.path.isdir(f"{w}/bronze.db/pid_documents"))   # the one holding Bronze
bronze_path = f"{WH}/bronze.db/pid_documents"

# 2) confirm Bronze has both formats now
spark.read.format("delta").load(bronze_path).groupBy("source_format").count().show()

In [ ]:
# 3) rebuild Silver over ALL Bronze rows (A + B), in THIS session
from silver.notebook import reconstruct
print(reconstruct(spark, bronze_path=bronze_path))

# 4) parity checks — Silver was written into this session, so query by name
spark.table("silver.silver_segments").groupBy("source_format").count().show()
spark.table("silver.silver_connections").groupBy("source_format", "flow_sense").count().show()
spark.table("silver.silver_components").groupBy("source_format", "is_valve").count().show()